In [1]:
import requests  # Untuk mengirim permintaan HTTP
from bs4 import BeautifulSoup  # Untuk mem-parsing HTML
import pandas as pd  # Untuk membuat DataFrame dan menyimpan ke CSV
import time  # Untuk memberikan jeda antar permintaan
import os # Untuk membuat direktori output

In [5]:
target_url_jobstreet = "https://id.jobstreet.com/id/Data-Science-jobs"
headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}
try:
    response = requests.get(target_url_jobstreet, headers=headers, timeout=15)
    response.raise_for_status()
    print("Permintaan berhasil, status code:", response.status_code)

    # Simpan HTML ke file untuk dianalisis
    with open("jobstreet_page.html", "w", encoding="utf-8") as f:
        f.write(response.text)
    print("HTML halaman telah disimpan ke jobstreet_page.html")

    soup = BeautifulSoup(response.content, 'html.parser')

    # --- GANTI SELECTOR DI BAWAH INI ---
    # Coba selector yang paling umum dulu, misalnya hanya tag <article>
    # atau tag <div> dengan class yang sangat umum untuk daftar item.
    # Ini hanya untuk melihat apakah kita bisa mendapatkan *sesuatu*.
    # job_containers = soup.find_all('article')
    # job_containers = soup.find_all('div', class_='INI_CLASS_YANG_ANDA_TEMUKAN_DARI_INSPEKSI')

    # Berdasarkan screenshot Anda, mari kita coba lagi dengan sedikit variasi atau lebih umum:
    # Percobaan 1: Artikel secara umum (mungkin terlalu banyak)
    # job_containers = soup.find_all('article')

    # Percobaan 2: Div yang memiliki atribut 'data-automation' yang berisi 'jobListing'
    # Ini lebih longgar daripada 'startswith'
    job_containers = soup.find_all(lambda tag: tag.name == 'article' and tag.has_attr('data-automation') and 'jobListing' in tag['data-automation'])

    # Percobaan 3: Cari div yang di dalamnya ada link dengan data-automation="jobTitle"
    # Ini pendekatan tidak langsung, jika kita tahu setiap job pasti punya jobTitle
    # job_titles_links = soup.find_all('a', attrs={'data-automation': 'jobTitle'})
    # job_containers = [link.find_parent('article') for link in job_titles_links] # atau find_parent('div')
    # job_containers = [container for container in job_containers if container] # Hapus None jika ada

    if job_containers:
        print(f"DITEMUKAN {len(job_containers)} kontainer dengan selector yang dicoba.")
        # Jika ini berhasil, Anda bisa melanjutkan dengan kode ekstraksi detail.
        # Untuk sekarang, kita hanya ingin memastikan kontainer utama ditemukan.
        # print(job_containers[0].prettify()) # Cetak HTML dari kontainer pertama untuk dilihat
    else:
        print("GAGAL menemukan kontainer lowongan dengan selector yang dicoba.")
        print("Silakan buka file 'jobstreet_page.html' yang telah disimpan,")
        print("dan bandingkan dengan Developer Tools browser Anda untuk menemukan selector yang tepat.")

except requests.exceptions.RequestException as e:
    print(f"Error saat mengirim permintaan HTTP: {e}")
except Exception as e:
    print(f"Terjadi error lain: {e}")

Permintaan berhasil, status code: 200
HTML halaman telah disimpan ke jobstreet_page.html
GAGAL menemukan kontainer lowongan dengan selector yang dicoba.
Silakan buka file 'jobstreet_page.html' yang telah disimpan,
dan bandingkan dengan Developer Tools browser Anda untuk menemukan selector yang tepat.


In [7]:
# ... (setelah soup = BeautifulSoup(...)) ...

        # --- SISIPKAN KODE DEBUGGING INI ---
        print("\n--- DEBUG: Mencoba beberapa selector umum untuk kontainer ---")

        # Percobaan 1: Semua tag <article>
        articles = soup.find_all('article')
        print(f"Ditemukan {len(articles)} tag <article>.")
        if articles:
            print("Contoh HTML <article> pertama:")
            print(articles[0].prettify()[:500]) # Cetak 500 karakter pertama dari HTML artikel pertama
            # Periksa apakah atribut data-automation ada dan polanya
            for i, art in enumerate(articles[:3]): # Cek 3 artikel pertama
                if art.has_attr('data-automation'):
                    print(f"Artikel {i} punya data-automation: {art['data-automation']}")
                else:
                    print(f"Artikel {i} TIDAK punya data-automation.")

        # Percobaan 2: Semua tag <div> dengan class yang mungkin relevan (ANDA HARUS ISI CLASSNYA)
        # Ganti 'class-yang-mungkin-relevan' dengan class yang Anda lihat di jobstreet_page.html
        # divs_with_class = soup.find_all('div', class_='class-yang-mungkin-relevan')
        # print(f"Ditemukan {len(divs_with_class)} tag <div> dengan class 'class-yang-mungkin-relevan'.")
        # if divs_with_class:
        #     print("Contoh HTML <div> pertama:")
        #     print(divs_with_class[0].prettify()[:500])

        # Percobaan 3: Cari elemen berdasarkan keberadaan link judul pekerjaan
        # (Jika setiap item pekerjaan PASTI memiliki link judul dengan pola tertentu)
        # title_links = soup.find_all('a', attrs={'data-automation': 'jobTitle'}) # Menggunakan selector judul dari revisi sebelumnya
        # print(f"Ditemukan {len(title_links)} link judul dengan data-automation='jobTitle'.")
        # if title_links:
        #     print("Mencoba mencari parent container dari link judul pertama...")
        #     # Coba cari parent <article> atau <div> yang membungkusnya
        #     parent_container_of_first_title = title_links[0].find_parent('article') # atau 'div'
        #     if parent_container_of_first_title:
        #         print("Contoh HTML parent container dari link judul pertama:")
        #         print(parent_container_of_first_title.prettify()[:500])
        #     else:
        #         print("Tidak ditemukan parent container <article> atau <div> untuk link judul pertama.")
        
        print("--- AKHIR DEBUG --- \n")
        
        # --- BARU KEMUDIAN COBA SELECTOR ANDA UNTUK job_containers ---
        # job_containers = soup.find_all(...) # Selector Anda yang sudah direvisi

IndentationError: unexpected indent (2800903372.py, line 4)